# MangoVision — Entrenamiento YOLOv8 en Google Colab

Notebook auxiliar para **EN-005 (baseline)** y, más adelante, **EN-006 (custom)**.
Pensado para correr en Colab con **runtime GPU** (Entorno de ejecución → Cambiar tipo → T4 GPU).

## Antes de empezar (una sola vez)

1. En tu PC, comprimí el dataset público a un zip:
   ```powershell
   Compress-Archive -Path "BackMangoVision\data\raw\public\mango-v1-yolov8\*" `
       -DestinationPath "$env:USERPROFILE\Desktop\mango-v1-yolov8.zip"
   ```
2. Subí ese `mango-v1-yolov8.zip` a tu Google Drive, por ejemplo a `MyDrive/MangoVision/`.
3. Ejecutá las celdas de abajo en orden.

> **EN-005** entrena un detector de **frutos** (`ripe`/`un_ripe`) — NO de enfermedades. Sirve como pre-entrenamiento de localización para EN-006 (ver `docs/sprints/sprint-3/evidencias/EN-005.md`).

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Instalar Ultralytics (YOLOv8)

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 3. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Parámetros (editá estas rutas)

In [ ]:
# --- Rutas en Drive ---
DRIVE_DIR    = '/content/drive/MyDrive/MangoVision'     # carpeta en tu Drive
DATASET_ZIP  = f'{DRIVE_DIR}/mango-v1-yolov8.zip'       # el zip que subiste
RESULTS_DIR  = f'{DRIVE_DIR}/runs'                       # donde se guardarán los resultados

# --- Hiperparámetros del run (EN-005) ---
TAG     = 'baseline-v0'
MODEL   = 'yolov8n.pt'   # nano: chico y rápido
EPOCHS  = 50
IMGSZ   = 640
BATCH   = 16             # bajá a 8 si hay Out-Of-Memory

import os
assert os.path.exists(DATASET_ZIP), f'No encuentro el zip en {DATASET_ZIP} — revisá DRIVE_DIR / que subiste el archivo.'
print('Zip OK:', DATASET_ZIP)

## 5. Descomprimir el dataset a /content

In [ ]:
import zipfile, os

DATA_ROOT = '/content/mango-v1-yolov8'
os.makedirs(DATA_ROOT, exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
    z.extractall(DATA_ROOT)

# Si el zip creó un nivel extra (mango-v1-yolov8/train...), detectarlo:
def find_splits(root):
    for cur, dirs, _ in os.walk(root):
        if {'train', 'valid'}.issubset(set(dirs)):
            return cur
    return root

DATA_ROOT = find_splits(DATA_ROOT)
print('Dataset root:', DATA_ROOT)
for s in ['train', 'valid', 'test']:
    p = os.path.join(DATA_ROOT, s, 'images')
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f'  {s}/images: {n}')

## 6. Generar data.yaml con rutas absolutas (Colab)

El `data.yaml` original de Roboflow usa rutas relativas (`../train/images`) que fallan en Colab. Lo regeneramos apuntando a rutas absolutas. Clases del baseline: `ripe` / `un_ripe`.

In [ ]:
yaml_path = '/content/data_colab.yaml'
yaml_text = f'''path: {DATA_ROOT}
train: train/images
val: valid/images
test: test/images

nc: 2
names: ['ripe', 'un_ripe']
'''
with open(yaml_path, 'w') as f:
    f.write(yaml_text)
print(yaml_text)

## 7. Entrenar

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    project='/content/runs',
    name=TAG,
    exist_ok=True,
    plots=True,
)

## 8. Validar en el test set (mAP)

In [ ]:
metrics = model.val(data=yaml_path, split='test')
print('mAP@0.5      :', round(float(metrics.box.map50), 4))
print('mAP@0.5:0.95 :', round(float(metrics.box.map), 4))

## 9. Guardar resultados en Drive

Copia el run completo (pesos + gráficos) a tu Drive para descargarlo a la PC.

In [ ]:
import shutil, os
run_dir = f'/content/runs/{TAG}'
dest = f'{RESULTS_DIR}/{TAG}'
os.makedirs(RESULTS_DIR, exist_ok=True)
if os.path.exists(dest):
    shutil.rmtree(dest)
shutil.copytree(run_dir, dest)
print('Resultados copiados a Drive:', dest)
print('Pesos finales:', f'{dest}/weights/best.pt')

## 10. De vuelta en tu PC (cerrar EN-005)

1. Descargá de Drive `runs/baseline-v0/weights/best.pt` y copialo a:
   ```
   BackMangoVision/ml/models/baseline-v0.pt
   ```
2. Descargá toda la carpeta `runs/baseline-v0/` y, desde el repo, extraé las evidencias:
   ```powershell
   python scripts/extract_yolo_artifacts.py --run-dir <ruta-descargada>/baseline-v0 --tag baseline-v0
   ```
   Eso copia `results.png`, `PR_curve.png`, `confusion_matrix.png`, etc. a
   `docs/sprints/sprint-3/evidencias/screenshots/baseline-v0/` y genera `SUMMARY.md`.
3. Actualizá [`EN-005.md`](../../docs/sprints/sprint-3/evidencias/EN-005.md) con el `mAP@0.5` del paso 8 y marchá el estado a ✅.

## Para EN-006 (cuando exista el dataset propio anotado)

- Subí a Drive el zip de `data/processed` (generado por `scripts/split_dataset.py`).
- Cambiá en la celda 4: `TAG='custom-v1'`, `DATASET_ZIP` al nuevo zip, y en la celda 6 usá las **5 clases** del catálogo (`sano, antracnosis, oidio, pudricion_peduncular, otras_lesiones`).
- En la celda 7, arrancá desde el baseline: `model = YOLO('/content/drive/MyDrive/MangoVision/runs/baseline-v0/weights/best.pt')` (transfer learning).